In [ ]:
!pip install pandas openpyxl
!pip install selenium
!pip install 2captcha-python

In [ ]:
import os
import logging
from time import sleep
from datetime import datetime

import pandas as pd

from twocaptcha import TwoCaptcha

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

URL_PROFESSORES_SIGAA = "https://sigaa.ufpi.br/sigaa/public/departamento/professores.jsf?id=144"
CAPTCHA2_API_KEY = "b53520e9ff8d77c6751932a1a9133abb"
LATES_SITE_KEY = "6LeDv6QUAAAAAP_ZK5AXrsNRQjxfbjgZLV5_YHyy"

CAMINHO_RAIZ = os.getcwd()

DATA_LOG = datetime.now().strftime("%Y_%m_%d")
CAMINHO_LOG = os.path.join(CAMINHO_RAIZ, f"logs")
os.makedirs(CAMINHO_LOG, exist_ok=True)
CAMINHO_LOG = os.path.join(CAMINHO_LOG, f"gdf_regular_guias_geral_{DATA_LOG}.log")

logging.basicConfig(filename=CAMINHO_LOG, format='%(asctime)s - %(message)s', level=logging.INFO, encoding='utf-8')

In [ ]:
def resolver_captcha(site_key, page_url):
    api_key = CAPTCHA2_API_KEY

    solver = TwoCaptcha(api_key)

    try:
        tempo_inicial = datetime.now()

        result = solver.recaptcha(
            sitekey=site_key,
            url=page_url,
            # invisible=1,
            # enterprise=1
        )

        logging.info(result["code"][:100])

        return result["code"]
    except Exception as e:
        logging.info(e)
    finally:
        tempo_final = datetime.now()

        logging.info(f"Tempo de resolução do captcha: {tempo_final - tempo_inicial}")

In [ ]:
def coletar_dados_lattes(driver, wait, dados_docente):
    try:    
        driver.get(dados_docente["Lattes"])

        url_atual = driver.current_url

        codigo_captcha = resolver_captcha(LATES_SITE_KEY, url_atual)

        driver.execute_script("""
            document.getElementById('tokenCaptchar').value = arguments[0];
        """, codigo_captcha)

        sleep(2)

        driver.execute_script("""
            document.getElementById('formulario').submit();
        """)

        sleep(2)

        categorias = driver.find_elements(By.CLASS_NAME, "title-wrapper")
        producoes_docente = []
        bancas_docente = []

        for categoria in categorias:
            titulo_categoria = categoria.text

            if "Produções" in titulo_categoria:
                # print(categoria.text, "\n")
                producoes = categoria.find_elements(By.CLASS_NAME, "layout-cell.layout-cell-11")
                for producao in producoes:
                    doi = producao.find_elements(By.CLASS_NAME, "icone-doi")
                    doi_url = doi[0].get_attribute("href") if doi else ""

                    producoes_docente.append({
                        "Título": producao.text.strip(),
                        "URL": doi_url
                    })
            elif "Bancas" in titulo_categoria:
                # print(categoria.text, "\n")
                bancas = categoria.find_elements(By.CLASS_NAME, "layout-cell.layout-cell-11")
                for banca in bancas:
                    doi = banca.find_elements(By.CLASS_NAME, "icone-doi")
                    doi_url = doi[0].get_attribute("href") if doi else ""

                    bancas_docente.append({
                        "Título": banca.text.strip(),
                        "URL": doi_url
                    })

        dados_docente["Produções"] = producoes_docente
        dados_docente["Bancas"] = bancas_docente
    except Exception as e:
        logging.info(f"Falha ao coletar dados do Lattes: {e}")

    return dados_docente

In [ ]:
def coletar_dados_sigaa(driver, wait):
    try:
        dados_docentes = []

        driver.get(URL_PROFESSORES_SIGAA)

        tabelas = wait.until(
            EC.presence_of_all_elements_located(
                (By.TAG_NAME, "tbody")
            )
        )

        for tabela in tabelas:
            for linha in tabela.find_elements(By.TAG_NAME, "tr"):
                colunas = linha.find_elements(By.TAG_NAME, "td")

                url_lattes = colunas[3].find_elements(By.LINK_TEXT, "Currículo Lattes")
                url_lattes = url_lattes[0].get_attribute("href") if url_lattes else ""

                dados_docentes.append({
                    "Foto": colunas[0].find_element(By.TAG_NAME, "img").get_attribute("src"),
                    "Nome": colunas[1].text.strip(),
                    "Formação": colunas[2].get_attribute("textContent").strip(),
                    "Lattes": url_lattes
                })
    except Exception as e:
        logging.info(f"Falha ao coletar dados do SIGAA: {e}")

    return dados_docentes

In [ ]:
def main():
    driver = webdriver.Chrome()

    wait = WebDriverWait(driver, timeout=30)

    dados_docentes = coletar_dados_sigaa(driver, wait)

    for indice_docente, dados_docente in enumerate(dados_docentes):
        logging.info(f"Executando para o docente {dados_docente['Nome']}: {indice_docente + 1} de {len(dados_docentes)}")

        dados_docente = coletar_dados_lattes(driver, wait, dados_docente)

        # Remover/comentar o break para coletar todos os dados
        break

    driver.quit()

    return dados_docentes

In [ ]:
dados_docentes = main()

In [ ]:
caminho_relatorio = os.path.join(CAMINHO_RAIZ, f"coleta_dados_doscentes_{DATA_LOG}.csv")

pd.DataFrame(dados_docentes).to_csv(caminho_relatorio, index=False)

In [ ]:
pd.DataFrame(dados_docentes)